# Project 2 — Personal Finance Dashboard
## Notebook 1: Exploratory Data Analysis

**Goal:** Analyse 12-month personal expense patterns — spending trends, budget adherence, category breakdown  
**Skills demonstrated:** pandas, matplotlib, seaborn, time-series analysis, budget variance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
df     = pd.read_csv('../data/expenses_synthetic.csv', parse_dates=['date'])
budget = pd.read_csv('../data/monthly_budgets.csv')

expenses = df[df['type'] == 'debit'].copy()
income   = df[df['type'] == 'credit'].copy()

print(f'Total transactions: {len(df):,}')
print(f'Total spend (2024): ${expenses.amount.sum():,.0f}')
print(f'Total income (2024): ${income.amount.sum():,.0f}')
print(f'Net savings: ${income.amount.sum() - expenses.amount.sum():,.0f}')

## 1. Monthly Spend Trend vs Income

In [ ]:
monthly_spend  = expenses.groupby('month')['amount'].sum().reset_index()
monthly_income = income.groupby('month')['amount'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_spend['month'], monthly_spend['amount'],
        marker='o', color='#e74c3c', lw=2.5, label='Total Spend')
ax.axhline(monthly_income['amount'].mean(), color='#2ecc71', linestyle='--',
           lw=2, label=f'Monthly Income (${monthly_income["amount"].mean():,.0f})')
ax.fill_between(monthly_spend['month'], monthly_spend['amount'],
                monthly_income['amount'].mean(), alpha=0.08, color='#e74c3c')
ax.set_title('Monthly Spend vs Income — 2024', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Amount ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('../tableau/screenshots/monthly_trend.png', bbox_inches='tight')
plt.show()

## 2. Spending by Category

In [ ]:
cat_totals = expenses.groupby('category')['amount'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
palette = sns.color_palette('Set2', len(cat_totals))
axes[0].barh(cat_totals.index[::-1], cat_totals.values[::-1], color=palette[::-1])
axes[0].set_title('Total Annual Spend by Category', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(cat_totals.values[::-1]):
    axes[0].text(v + 100, i, f'${v:,.0f}', va='center', fontsize=8)

# Donut chart
wedges, texts, autotexts = axes[1].pie(
    cat_totals.values, labels=cat_totals.index,
    autopct='%1.1f%%', colors=palette,
    startangle=140, pctdistance=0.75
)
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
axes[1].add_patch(centre_circle)
axes[1].set_title('Category Share of Total Spend', fontweight='bold')

plt.tight_layout()
plt.savefig('../tableau/screenshots/category_breakdown.png', bbox_inches='tight')
plt.show()

## 3. Budget vs Actual Variance

In [ ]:
annual_spend  = expenses.groupby('category')['amount'].sum().reset_index()
annual_budget = budget.groupby('category')['budget_amount'].sum().reset_index()
variance_df   = annual_spend.merge(annual_budget, on='category')
variance_df['variance']     = variance_df['amount'] - variance_df['budget_amount']
variance_df['variance_pct'] = (variance_df['variance'] / variance_df['budget_amount'] * 100).round(1)
variance_df = variance_df.sort_values('variance', ascending=True)

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in variance_df['variance']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(variance_df['category'], variance_df['variance'], color=colors)
ax.axvline(0, color='black', lw=1)
ax.set_title('Budget Variance by Category — 2024\n(Red = over budget | Green = under budget)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Variance ($)')
for bar, pct in zip(bars, variance_df['variance_pct']):
    x = bar.get_width()
    ax.text(x + (20 if x >= 0 else -20), bar.get_y() + bar.get_height()/2,
            f'{pct:+.1f}%', va='center', ha='left' if x >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../tableau/screenshots/budget_variance.png', bbox_inches='tight')
plt.show()

print('\nBudget Variance Summary:')
print(variance_df[['category', 'budget_amount', 'amount', 'variance', 'variance_pct']].to_string(index=False))

## 4. Top Merchants by Spend

In [ ]:
top_merchants = expenses.groupby('merchant')['amount'].sum().nlargest(12).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_merchants['merchant'][::-1], top_merchants['amount'][::-1],
        color=sns.color_palette('Blues_r', len(top_merchants)))
ax.set_title('Top 12 Merchants by Annual Spend', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('../tableau/screenshots/top_merchants.png', bbox_inches='tight')
plt.show()

## EDA Summary

| KPI | Value |
|---|---|
| Total 2024 Income | $66,000 |
| Total 2024 Spend | ~$49,000 |
| Savings Rate | ~26% |
| Over-budget categories | Dining Out, Shopping |
| Seasonal spike | Nov/Dec (Shopping +20%) |

**Next:** Connect `expenses_synthetic.csv` + `monthly_budgets.csv` to Tableau and build the interactive dashboard